In [1]:
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from flax.core import unfreeze, freeze

from pyscf import gto, scf, ao2mo, cc

import importlib

import wavefunctions, hamiltonian, trajectory, qc

import time

/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


## 3D Cubic Lattice

In [2]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (7,7)
dim = 3

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:

    # cubic lattice
    lattice = wavefunctions.computeLattice(N, r_ws, dim)
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 81


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	0.6065345994595972
CCSD:    	0.5722091011461401
CCSD(T): 	0.5709088539648196

2.0 81
UHF:     	0.02303901700811422
CCSD:    	-0.0045026189248119275
CCSD(T): 	-0.0075574314571241336

5.0 81
UHF:     	-0.05933699180813085
CCSD:    	-0.07537580627315191
CCSD(T): 	-0.08173105460520384

10.0 81
UHF:     	-0.0447290662524232
CCSD:    	-0.051121211330323935
CCSD(T): 	-0.054874072960566664

20.0 81
UHF:     	-0.027546777313475584
CCSD:    	-0.029773858273465077
CCSD(T): 	-0.030482519101926595

50.0 81
UHF:     	-0.012688641445035937
CCSD:    	-0.013363274167462904
CCSD(T): 	-0.01352710477194195



## 3D Skewed Lattice

In [3]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (7,7)
dim = 3

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:

    # non-cubic lattice
    lattice = wavefunctions.computeLattice(
        N, r_ws, dim,
        basis_matrix=jnp.eye(dim) + 0.2
    )
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=9.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 77


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	0.7106951060045444
CCSD:    	0.6758708172366983
CCSD(T): 	0.6743916576370231

2.0 77
UHF:     	0.04986813295761276
CCSD:    	0.022152908525836932
CCSD(T): 	0.018806046856451358

5.0 77
UHF:     	-0.05844797835128625
CCSD:    	-0.07123211052121071
CCSD(T): 	-0.07521993702427901

10.0 77
UHF:     	-0.04555810075470089
CCSD:    	-0.05089506011171604
CCSD(T): 	-0.05266261427143737

20.0 77
UHF:     	-0.027642326131932876
CCSD:    	-0.0299717118016401
CCSD(T): 	-0.030713271788065107

50.0 77
UHF:     	-0.01263701087699934
CCSD:    	-0.013325060132121608
CCSD(T): 	-0.013501758165207864



## 2D Square Lattice

In [4]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (5,5)
dim = 2

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]:

    # square lattice
    lattice = wavefunctions.computeLattice(N, r_ws, dim)
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 45


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	-0.12461069820167739
CCSD:    	-0.2170701975015133
CCSD(T): 	-0.22529965128280108

2.0 45
UHF:     	-0.20102896682381713
CCSD:    	-0.2526666864239526
CCSD(T): 	-0.2662707501385517

5.0 45
UHF:     	-0.1325183134413145
CCSD:    	-0.1448578651853233
CCSD(T): 	-0.1485059243764377

10.0 45
UHF:     	-0.07899996057038096
CCSD:    	-0.0830056019209361
CCSD(T): 	-0.08420768211867509

20.0 45
UHF:     	-0.04376239156809152
CCSD:    	-0.04505545024336648
CCSD(T): 	-0.04530737762609049

50.0 45
UHF:     	-0.01870903574138334
CCSD:    	-0.019110902628332392
CCSD(T): 	-0.019172474811894737

100.0 45
UHF:     	-0.00956344279956673
CCSD:    	-0.009766627845080653
CCSD(T): 	-0.009793401019454967



## 2D Skewed Lattice

In [5]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (5,5)
dim = 2

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]:

    # non-square lattice
    lattice = wavefunctions.computeLattice(
        N, r_ws, dim,
        basis_matrix=jnp.eye(dim) + 0.2
    )
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 51


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	-0.10373934436340501
CCSD:    	-0.1896679897680123
CCSD(T): 	-0.2012662727074625

2.0 51
UHF:     	-0.2036008103185281
CCSD:    	-0.2487966372763438
CCSD(T): 	-0.25869541207413177

5.0 51
UHF:     	-0.13188002331044119
CCSD:    	-0.1447658367826886
CCSD(T): 	-0.14830789451418508

10.0 51
UHF:     	-0.07924255426211918
CCSD:    	-0.08316312739193275
CCSD(T): 	-0.0841846912825896

20.0 51
UHF:     	-0.04412430492314341
CCSD:    	-0.04540407294111003
CCSD(T): 	-0.045589735946495624

50.0 51
UHF:     	-0.01883649441842725
CCSD:    	-0.019126857248887574
CCSD(T): 	-0.01920251983292041

100.0 51
UHF:     	-0.009654876622199448
CCSD:    	-0.009843755644640332
CCSD(T): 	-0.009865357457394917

